# Time series Notebook
The aim of this notebook is to compare the timeseries of topic inside/between clusters.

## Load libraries

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
from bertopic import BERTopic

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data and create the timeseries dataset

In [9]:
cluster_df = pd.read_parquet('community_detection_results_4B_0_0_5.parquet')

In [4]:
cluster_df['Cluster Label'].unique()

array(['Multidrug-resistant TB Research', 'Zoonotic Disease Surveillance',
       'Mosquito-borne Diseases', 'Food Safety and Pathogen Risk',
       'Antibiotic Resistance Dynamics',
       'Influenenza Viruses and Pandemics', 'Monkeypox Outbreak Response',
       'Bovine TB and Culling Controversy', 'SARS-CoVs Variant Dynamics',
       'Infection Control Practices', 'Vaccine Safety Monitoring',
       'Air Quality Policy Impact', 'Malaria Prevention Strategies',
       'Pandemic Response Strategies', 'Zika and Arbovirus Research',
       'Pandemic Impact on Education and Health',
       'HIV and Related Viral Dynamics', 'Polio Prevention Strategies',
       'Sexually Transmitted Infections',
       'Vaccine Immunity and Outbreaks',
       'Antiviral Therapies for COVID-19', 'Zoonotic Virus Emergences',
       'HIV Care and Epidemic Management', 'Vaccine Information Dynamics',
       'Maternal and Child Health', 'Antiviral Drug Resistance Policies',
       'Diagnostic Technologies and 

In [19]:
model_list = ['scopus','science_news','the_guardian']

In [20]:
timeseries_df = pd.DataFrame()

In [21]:
cfg_dict = cfg.MAGAZINE_CONFIG['the_guardian']
model_data = np.load(cfg_dict['OUTPUT_PATH'],allow_pickle=True)

In [22]:
pd.DataFrame({'id': model_data['id'],
             'summary':model_data['text']}).to_csv('summary.csv',sep='ç',encoding='utf-8')

In [23]:
for model in model_list:
    
    cfg_dict = cfg.MAGAZINE_CONFIG[model]
    bertopic_model = BERTopic.load(cfg_dict['REFERENCE_MODEL'])

    model_data = np.load(cfg_dict['OUTPUT_PATH'],allow_pickle=True)

    dataset = pd.read_parquet(cfg_dict['DATASET_PATH'])
    
    tmp_df = bertopic_model.get_document_info(model_data['text'])['CustomName'].reset_index()
    tmp_df = tmp_df.drop(columns='index')

    tmp_df['id'] =  list(model_data['id'])
    tmp_df = tmp_df.rename(columns={'CustomName':'Topic Label'})
    
    tmp_df = tmp_df.merge(cluster_df,on='Topic Label')

    columns_to_mantain = ['id','publicationDate']
    columns_to_drop = [column for column in dataset.columns if column not in columns_to_mantain ]

    tmp_df = tmp_df.merge(dataset.drop(columns=columns_to_drop),on='id')

    timeseries_df = pd.concat([timeseries_df,tmp_df])


2026-04-27 11:31:02,308 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-04-27 11:31:16,817 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-04-27 11:31:18,222 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


## Select and plot clusters

In [24]:
cluster_df[['Cluster','Cluster Label']].value_counts().reset_index().sort_values(by='Cluster') 

,Cluster,Cluster Label,count
0,1,Pandemic Response Strategies,17
1,2,Antibiotic Resistance Dynamics,12
4,3,Influenenza Viruses and Pandemics,9
2,4,Zoonotic Virus Emergences,10
10,5,Zoonotic Disease Surveillance,7
6,6,Pandemic Impact on Education and Health,8
13,7,Mosquito-borne Diseases,6
3,9,Vaccine Immunity and Outbreaks,10
8,10,Microbial and Genetic Therapies,7
15,11,HIV Care and Epidemic Management,5


In [25]:
#Covid
#cluster_list = [1,7,11,13,17,19,25,30,36,37,38]
# HIV
#cluster_list = [8,18,32]
# Influenza
#cluster_list = [4,29]
#Ebola
#cluster_list= [9]
#Malaria
#cluster_list = [12]
# Mpox
#cluster_list = [28]
# HAI and AMR


#4b
#covid
#cluster_list = [6,15,16,19,20,42,40,22,34]
#vaccine
#cluster_list = [9,27,32,45,46,47,48]
cluster_list = [1]

In [26]:
timeseries_df_analysis = timeseries_df[ timeseries_df['Cluster'].isin(cluster_list) ]

In [27]:
if len(cluster_list) > 1:
    # Settare un identificativo per Cluster
    if 'Cluster Label' in timeseries_df_analysis.columns:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster Label']
    else:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster']
else:
    # Settare un identificativo per Topic Label
    timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Topic Label']


In [28]:
import polars as pl

timeseries_df_analysis_polars = pl.from_pandas(timeseries_df_analysis)

In [29]:
#period = '1mo'
period = '1y'

In [30]:
timeseries_df_analysis_polars = timeseries_df_analysis_polars.with_columns(
    pl.col('publicationDate').dt.truncate(period).alias('timestamp'),
)

In [31]:
partial_timeseries_df_analysis_polars = timeseries_df_analysis_polars.group_by(['Graph_label','timestamp']).agg(
        pl.len().alias('occurrence')
    ).sort(by=['timestamp','Graph_label'],descending=False)

In [32]:
dates = pl.date_range(
    start= partial_timeseries_df_analysis_polars.select(pl.col("timestamp").min()).item(),
    end=  partial_timeseries_df_analysis_polars.select(pl.col("timestamp").max()).item(),
    interval= period,
    eager = True
)

In [33]:
topics = ( 
    partial_timeseries_df_analysis_polars.select(
        pl.col('Graph_label').unique().sort()).to_series()
)

In [34]:
grid = (
    pl.DataFrame({'Graph_label':topics}).join(pl.DataFrame({'timestamp':dates}),how='cross')
)

In [35]:
partial_timeseries_df_analysis_polars = partial_timeseries_df_analysis_polars.with_columns(
        pl.col("timestamp").dt.date().alias("timestamp")
)

In [36]:
complete_timeseries_df_analysis_polars  = (
    grid.join(partial_timeseries_df_analysis_polars,on=['Graph_label','timestamp'],how='left')
    .with_columns(pl.col('occurrence').fill_null(0))
    .sort(by=['timestamp','Graph_label'],descending=False)
)

In [37]:
complete_timeseries_df_analysis = complete_timeseries_df_analysis_polars.to_pandas()

In [38]:
import plotly.express as px

if len(cluster_list) > 1:
    legend_title = "Cluster"
    graph_title = "Cluster time series"
else:
    legend_title = "Topics"
    graph_title= "Topics time series"

fig = px.line(
    complete_timeseries_df_analysis,
    x="timestamp",
    y="occurrence",
    color="Graph_label",
    markers=True,
    title=graph_title
)

fig.update_layout(
    # Titolo
    title={
        'text': graph_title,
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'family': 'Arial, sans-serif'}
    },
    
    # Assi
    xaxis_title="Time",
    yaxis_title="# Articles",
    xaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black'
    ),
    yaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black'
    ),    
    legend=dict(
        title=legend_title,
        orientation="h",  
        yanchor="top",
        y=-0.15, 
        xanchor="center",
        x=0.5,  
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    ),
    
    
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,
    height=600,
    
    
    margin=dict(l=80, r=80, t=80, b=120),
    
    
    hovermode='x unified'
)


fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=6)
)

fig.show()

## New graph - Splitted per model and normalized per model/year

### Load the dataset that we use to normalize 

In [39]:
model_articles_per_year = pd.read_parquet('model_articles_per_year.parquet')

### Create the timeseries datasets

Choose which clusters show

In [246]:
#cluster_list = list(range(0,10))
#cluster_list = list(range(10,20))
#cluster_list = list(range(20,30))
#cluster_list = list(range(30,45))
#Covid
#cluster_list = [6,15,16,19,20,42,40,22,34]
#vaccine
#cluster_list = [9,27,32,45,46,47,48]
# school
#cluster_list = [6]
cluster_list = [3]

In [247]:
timeseries_df_analysis = timeseries_df[ (timeseries_df['Cluster'].isin(cluster_list)) & (timeseries_df['publicationDate'].dt.year <= 2025)  ]

In [248]:
timeseries_df_analysis['Model'] = timeseries_df_analysis['Model'].apply( lambda x: x.title().replace('_',' '))

Set graph labels

In [249]:
if len(cluster_list) > 1:
    # Settare un identificativo per Cluster
    if 'Cluster Label' in timeseries_df_analysis.columns:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster Label']
    else:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster']
else:
    # Settare un identificativo per Topic Label
    timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Topic Label']

Choose the granularity of time period

In [250]:
period = '1y'
#period = '1mo'

Truncate date based on the time period

In [251]:
timeseries_df_analysis_polars = pl.from_pandas(timeseries_df_analysis).with_columns(
    pl.col('publicationDate').dt.truncate(period).alias('timestamp'),
)

In [252]:
partial_timeseries_df_analysis_polars = timeseries_df_analysis_polars.group_by(['Graph_label','Model','timestamp']).agg(
        pl.len().alias('occurrence')
    ).sort(by=['timestamp','Model','Graph_label'],descending=False)

In [253]:
dates = pl.date_range(
    start= partial_timeseries_df_analysis_polars.select(pl.col("timestamp").min()).item(),
    end=  partial_timeseries_df_analysis_polars.select(pl.col("timestamp").max()).item(),
    interval= period,
    eager = True
)

In [254]:
topics = ( 
    partial_timeseries_df_analysis_polars.select(['Graph_label','Model']).unique()
)

In [255]:
grid = (
    topics.join(pl.DataFrame({'timestamp':dates}),how='cross')
)

In [256]:
partial_timeseries_df_analysis_polars = partial_timeseries_df_analysis_polars.with_columns(
        pl.col("timestamp").dt.date().alias("timestamp")
)

In [257]:
complete_timeseries_df_analysis_polars  = (
    grid.join(partial_timeseries_df_analysis_polars,on=['Graph_label','Model','timestamp'],how='left')
    .with_columns(pl.col('occurrence').fill_null(0))
    .sort(by=['timestamp','Model','Graph_label'],descending=False)
)

In [258]:
complete_timeseries_df_analysis = complete_timeseries_df_analysis_polars.to_pandas()

### Params to decide the time granularity that will be displayed on graph

In [259]:
if period == '1y':
    how = 'frequency_per_year'
    complete_timeseries_df_analysis = complete_timeseries_df_analysis.merge(model_articles_per_year,on=['Model','timestamp'])
    complete_timeseries_df_analysis['frequency_per_year'] = (complete_timeseries_df_analysis['occurrence'] / complete_timeseries_df_analysis['Articles']) * 100_000
else:
    how = 'occurrence'

In [260]:
complete_timeseries_df_analysis

,Graph_label,Model,timestamp,occurrence,Articles,frequency_per_year
0,Human Transmission of Avian Influenza,Science News,2001-01-01,2,1128,177.304965
1,Avian H5N1 Polymerase Adaptation,Scopus,2001-01-01,1,1241646,0.080538
2,Avian-origin Influenza Virus Strains,Scopus,2001-01-01,0,1241646,0.000000
3,H7N9 Avian Influenza Virus,Scopus,2001-01-01,0,1241646,0.000000
4,H9N2 Avian Influenza Virus,Scopus,2001-01-01,2,1241646,0.161077
...,...,...,...,...,...,...
220,H9N2 Avian Influenza Virus,Scopus,2025-01-01,19,4304715,0.441376
221,Highly Pathogenic Avian Influenza Viruses,Scopus,2025-01-01,160,4304715,3.716855
222,Swine Origin Influenza A Viruses,Scopus,2025-01-01,50,4304715,1.161517
223,H1N1 Swine Flu Pandemic,The Guardian,2025-01-01,0,72085,0.000000


### Graph plot

In [261]:
import plotly.express as px
import pandas as pd

models  = list(pd.unique(complete_timeseries_df_analysis["Model"]))

if len(cluster_list) > 1:
    legend_title = "Cluster"
    graph_title = "Cluster time series"
else:
    legend_title = "Topics"
    graph_title= "Topics time series"

fig = px.line(
    complete_timeseries_df_analysis,
    x="timestamp",
    y=how,
    color="Graph_label",
    markers=True,
    line_group="Graph_label",
    facet_row="Model",
    title=graph_title,
    facet_row_spacing=0.12
)


fig.update_layout(
    title={
        'text': graph_title,
        'x': 0.5,
        'y' : 0.98,
        'xanchor': 'center',
        'font': {'size': 20, 'family': 'Arial, sans-serif'}
    },
    legend=dict(
        title=legend_title,
        orientation="h",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,
    height=700,
    margin=dict(l=80, r=80, t=80, b=120),
    hovermode='x unified'
)


for r in range(1, len(models) + 1):
    fig.update_xaxes(
        title_text="Time" if r == 1 else None,  
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black',
        showticklabels=True,   
        ticks="outside"
    )

for r in range(1, len(models) + 1):
    fig.update_yaxes(
        title_text="# Articles" if r == 2 and how=='occurrence' else '# Articles per 100k' if r == 2 and how=='frequency_per_year' else  None,
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black',
    )

for ann in fig.layout.annotations:
    if ann.text.startswith("Model="):
        ann.text = ann.text.replace("Model=", "") 
        ann.x = 0.5                                
        ann.xanchor = "center"
        ann.y += 0.15                              
        ann.yanchor = "bottom"
        ann.font = dict(size=15, family="Arial, sans-serif")
        ann.textangle = 0 


fig.update_yaxes(matches=None)


fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=6)
)

fig.show()


In [262]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import pandas as pd

models = list(pd.unique(complete_timeseries_df_analysis["Model"]))
labels = list(pd.unique(complete_timeseries_df_analysis["Graph_label"]))

y_axis_labels = []

for r in range(1, len(models) + 1):
    if r == 2:
        y_axis_labels.append(
            "<b># Articles</b>"
            if how == "occurrence"
            else "<b># Articles per 100k</b>"
            if how == "frequency_per_year"
            else f"<b>{how}</b>"
        )
    else: 
        y_axis_labels.append('')

y_axis_labels.append("<b># Articles</b>")

if len(cluster_list) > 1:
    legend_title = "Clusters"
    hist_subplot_title = 'cluster'
    graph_title = "Clusters time series"
else:
    legend_title = "Topics"
    hist_subplot_title = 'topic'
    graph_title = "Topics time series"


# Colori coerenti tra lineplot e istogramma
palette = px.colors.qualitative.Plotly
color_map = {
    label: palette[i % len(palette)]
    for i, label in enumerate(labels)
}


# Una riga per ogni Model + una riga finale per l'istogramma comune
n_rows = len(models) + 1

row_heights = [0.75 / len(models)] * len(models) + [0.25]

subplot_titles = ["<b>"+str(model)+"</b>" for model in models] + [f"<b>Overall {hist_subplot_title} volume over time</b>"]


fig = make_subplots(
    rows=n_rows,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.15,
    row_heights=row_heights,
    subplot_titles=subplot_titles
)


shown_legend = set()

# -------------------------
# 1. Lineplot per ogni Model
# -------------------------
for i, model in enumerate(models):
    df_model = complete_timeseries_df_analysis[
        complete_timeseries_df_analysis["Model"] == model
    ]

    row = i + 1

    for label in labels:
        df_label = df_model[df_model["Graph_label"] == label]

        if df_label.empty:
            continue

        show_legend = label not in shown_legend

        fig.add_trace(
            go.Scatter(
                x=df_label["timestamp"],
                y=df_label[how],
                mode="lines+markers",
                name=str(label),
                legendgroup=str(label),
                showlegend=show_legend,
                line=dict(
                    width=2.5,
                    color=color_map[label]
                ),
                marker=dict(size=6),
                hovertemplate=(
                    "Time=%{x}<br>"
                    f"{how}=%{{y}}<br>"
                    f"{legend_title}={label}<extra></extra>"
                )
            ),
            row=row,
            col=1
        )

        shown_legend.add(label)


# --------------------------------------
# 2. Istogramma stacked comune ai modelli
# --------------------------------------

# Se vuoi un istogramma comune, devi aggregare ignorando Model.
# Qui sommo occurrence per timestamp e Graph_label.
hist_df = (
    complete_timeseries_df_analysis
    .groupby(["timestamp", "Graph_label"], as_index=False)["occurrence"]
    .sum()
)

hist_row = n_rows

for label in labels:
    df_label = hist_df[hist_df["Graph_label"] == label]

    if df_label.empty:
        continue

    fig.add_trace(
        go.Bar(
            x=df_label["timestamp"],
            y=df_label["occurrence"],
            name=str(label),
            legendgroup=str(label),
            showlegend=False,
            marker_color=color_map[label],
            opacity=0.85,
            hovertemplate=(
                "Time=%{x}<br>"
                "Articles=%{y}<br>"
                f"{legend_title}={label}<extra></extra>"
            )
        ),
        row=hist_row,
        col=1
    )


fig.update_layout(
    title={
        "text": "<b>"+graph_title+"</b>",
        "x": 0.5,
        "y": 0.98,
        "xanchor": "center",
        "font": {"size": 20, "family": "Arial, sans-serif"}
    },
    legend=dict(
        title=legend_title,
        orientation="h",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    width=1250,
    height=850,
    margin=dict(l=80, r=80, t=100, b=140),
    hovermode="x unified",
    barmode="stack"
)


# -------------------------
# Formattazione assi X e Y
# -------------------------
for r in range(1, n_rows + 1):
    fig.update_xaxes(
        title_text="<b>Time</b>" if r == n_rows else None,
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor="lightgray",
        showline=True,
        linewidth=2,
        linecolor="black",
        showticklabels=True,
        ticks="outside"
    )

    fig.update_yaxes(
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor="lightgray",
        showline=True,
        linewidth=2,
        linecolor="black",
        title_standoff=40,
        ticklabelstandoff=8,
        automargin=False,
        #ticklabelposition="inside",
    )


fig.update_yaxes(
        title_text=None,
        row=2,
        col=1
)



for r, label in enumerate(y_axis_labels, start=1):
    axis_name = "yaxis" if r == 1 else f"yaxis{r}"
    y_domain = fig.layout[axis_name].domain
    y_mid = sum(y_domain) / 2

    fig.add_annotation(
        text=label,
        xref="paper",
        yref="paper",
        x=-0.08,
        y=y_mid,
        textangle=-90,
        showarrow=False,
        font=dict(size=15, family="Arial, sans-serif"),
        xanchor="center",
        yanchor="middle"
    )
    
fig.update_layout(
    margin=dict(l=120, r=80, t=100, b=140)
)


fig.show()